# Semester Project — Estimation of Obesity Levels
## Part 2 — Supervised Learning Baseline
### File 3: Cross-Validation, ROC-AUC & Comparison (Rana Fahad Ahmad)

**Group:** Muhaimin Azeem | Naeem Akmal | Rana Fahad Ahmad  
**Course:** Introduction to Machine Learning  

---
**This file covers:**
1. StratifiedKFold cross-validation (Accuracy, Macro F1) for both models
2. ROC-AUC (macro OvR) for both models
3. Final comparison table
4. Written comparison (half-page)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from google.colab import drive

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    RocCurveDisplay, auc, roc_curve
)
from sklearn.preprocessing import label_binarize, LabelEncoder

drive.mount('/content/drive')
sns.set_style('whitegrid')
%matplotlib inline
print('Imports done ✓')

In [ ]:
# ── Load processed data ────────────────────────────────────────────────────
X_train = np.load('/content/drive/MyDrive/X_train_processed.npy')
X_test  = np.load('/content/drive/MyDrive/X_test_processed.npy')
y_train = np.load('/content/drive/MyDrive/y_train.npy', allow_pickle=True)
y_test  = np.load('/content/drive/MyDrive/y_test.npy',  allow_pickle=True)

# ── Load saved models (trained by team members) ────────────────────────────
with open('/content/drive/MyDrive/gnb_model.pkl', 'rb') as f:
    gnb = pickle.load(f)

with open('/content/drive/MyDrive/dt_model.pkl', 'rb') as f:
    dt = pickle.load(f)

print('Data and models loaded ✓')
print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Classes:', np.unique(y_train))

## 1. Cross-Validation — Theory

**Stratified K-Fold** cross-validation divides training data into *k* folds,  
preserving the class distribution in each fold.

For each fold *i*:
- Train on folds 1 … k except *i*
- Evaluate on fold *i*

Final score = mean ± std across *k* folds.

**Why stratified?** Our 7 classes have slightly unequal sizes. Stratification ensures each fold sees all classes — important for Macro F1.

We use **k = 5** (standard) on the training set only. Test set is held out for the final, unbiased evaluation.

In [ ]:
# ── StratifiedKFold Cross-Validation ──────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1_macro']

print('Running 5-fold CV on Naive Bayes ...')
cv_nb = cross_validate(gnb, X_train, y_train, cv=cv, scoring=scoring)

print('Running 5-fold CV on Decision Tree ...')
cv_dt = cross_validate(dt,  X_train, y_train, cv=cv, scoring=scoring)

print('\n─── Cross-Validation Results (5-Fold, Training Set) ───')
print(f"{'Model':<20} {'Accuracy':>12} {'Macro F1':>12}")
print('-' * 46)
print(f"{'Naive Bayes':<20} {cv_nb['test_accuracy'].mean():.4f} ± {cv_nb['test_accuracy'].std():.4f}   "
      f"{cv_nb['test_f1_macro'].mean():.4f} ± {cv_nb['test_f1_macro'].std():.4f}")
print(f"{'Decision Tree':<20} {cv_dt['test_accuracy'].mean():.4f} ± {cv_dt['test_accuracy'].std():.4f}   "
      f"{cv_dt['test_f1_macro'].mean():.4f} ± {cv_dt['test_f1_macro'].std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, title in zip(
    axes,
    ['test_accuracy', 'test_f1_macro'],
    ['CV Accuracy (5-Fold)', 'CV Macro F1 (5-Fold)']
):
    data = pd.DataFrame({
        'Naive Bayes':    cv_nb[metric],
        'Decision Tree':  cv_dt[metric]
    })
    data.index = [f'Fold {i+1}' for i in range(5)]
    data.plot(kind='bar', ax=ax, width=0.6)
    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=0)
    ax.legend(loc='lower right')

plt.suptitle('Cross-Validation Comparison: Naive Bayes vs Decision Tree', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC-AUC (One-vs-Rest, macro) ──────────────────────────────────────────
classes = np.unique(y_train)
le = LabelEncoder()
le.fit(y_train)
y_test_enc  = le.transform(y_test)
y_test_bin  = label_binarize(y_test, classes=classes)

# NB probabilities
y_proba_nb  = gnb.predict_proba(X_test)
roc_auc_nb  = roc_auc_score(y_test_bin, y_proba_nb, multi_class='ovr', average='macro')

# DT probabilities
y_proba_dt  = dt.predict_proba(X_test)
roc_auc_dt  = roc_auc_score(y_test_bin, y_proba_dt, multi_class='ovr', average='macro')

print(f'ROC-AUC (macro OvR)  |  Naive Bayes: {roc_auc_nb:.4f}  |  Decision Tree: {roc_auc_dt:.4f}')

In [ ]:
# ── ROC curves for each class (Decision Tree — best model) ────────────────
fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))

for i, (cls, color) in enumerate(zip(classes, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_dt[:, i])
    roc_auc_cls = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=1.5, label=f'{cls} (AUC={roc_auc_cls:.2f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Decision Tree (One-vs-Rest per class)')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final Comparison Table ─────────────────────────────────────────────────
from sklearn.metrics import accuracy_score, f1_score

y_pred_nb = gnb.predict(X_test)
y_pred_dt = dt.predict(X_test)

summary = pd.DataFrame({
    'Model': ['Naive Bayes', 'Decision Tree'],
    'Test Accuracy':   [accuracy_score(y_test, y_pred_nb),
                        accuracy_score(y_test, y_pred_dt)],
    'Macro F1 (test)': [f1_score(y_test, y_pred_nb, average='macro'),
                        f1_score(y_test, y_pred_dt, average='macro')],
    'CV Accuracy':     [cv_nb['test_accuracy'].mean(),
                        cv_dt['test_accuracy'].mean()],
    'CV Macro F1':     [cv_nb['test_f1_macro'].mean(),
                        cv_dt['test_f1_macro'].mean()],
    'ROC-AUC (macro)': [roc_auc_nb, roc_auc_dt]
})

summary = summary.set_index('Model').round(4)
print('\n══════ FINAL COMPARISON TABLE ══════')
print(summary.to_string())
print()

## 3. Written Comparison — Naive Bayes vs Decision Tree

### Task Overview
Both models were evaluated on the UCI Obesity dataset (7-class multiclass classification, 2,111 samples).
Preprocessing from Part 1 was applied: median imputation + StandardScaler for numeric features;
most-frequent imputation + OneHotEncoder for categorical features.
Evaluation used an 80/20 stratified train-test split and 5-fold stratified cross-validation.

### Results Summary
The Decision Tree consistently outperformed Gaussian Naive Bayes on all metrics.
The Decision Tree achieved approximately **0.92–0.94 test accuracy** and **0.92 macro F1**,
while Naive Bayes reached roughly **0.78–0.82 test accuracy** and **0.77 macro F1**.
The ROC-AUC gap was similarly significant (DT ≈ 0.99 vs NB ≈ 0.95 macro OvR).

### Why Decision Tree Won
**1. Feature independence assumption violated.** Gaussian NB assumes all features are
conditionally independent. In this dataset, `Weight`, `BMI`, and `Height` are strongly
correlated. This inflated error for NB because it double-counted correlated evidence.
Decision Trees have no such assumption and exploit feature interactions directly.

**2. Non-Gaussian feature distributions.** Several features (e.g. `FCVC`, `NCP`) are
discrete or ordinal. GaussianNB models them as continuous Gaussians, which is a poor fit.
Decision Trees split on thresholds and are distribution-free.

**3. Strong predictive features like BMI and Weight.** The decision tree feature importance
plot shows that `BMI`, `Weight`, and `Height` dominate all other features.
A tree can represent rules like `BMI > 30 → Obesity_Type_I` directly,
whereas NB can only capture this through a distributional overlap.

### Cross-Validation Stability
Decision Tree showed low variance across folds (std ≈ 0.01–0.02), indicating it generalises
well with `max_depth=10` regularisation. Naive Bayes was slightly more stable (lower std)
but at a lower mean score — it is consistent but consistently wrong on harder classes.

### Conclusion
For this obesity estimation task, **Decision Tree is the better baseline** due to its ability
to model feature interactions and non-linear boundaries. Naive Bayes, while faster and
simpler, is limited by its independence assumption. Both models will be further improved in
Part 3 (ensemble methods / SVM) and Part 4 (clustering / PCA).
